In [5]:
class AVLNode:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.height = 1

def get_height(node):
    return node.height if node else 0

def update_height(node):
    node.height = 1 + max(get_height(node.left), get_height(node.right))

def get_balance(node):
    return get_height(node.left) - get_height(node.right) if node else 0

def right_rotate(y):
    x = y.left
    T2 = x.right
    x.right = y
    y.left = T2
    update_height(y)
    update_height(x)
    return x

def left_rotate(x):
    y = x.right
    T2 = y.left
    y.left = x
    x.right = T2
    update_height(x)
    update_height(y)
    return y

def avl_insert(node, key, rotation_log=None):
    if not node:
        return AVLNode(key)
    if key < node.val:
        node.left = avl_insert(node.left, key, rotation_log)
    elif key > node.val:
        node.right = avl_insert(node.right, key, rotation_log)
    else:
        return node

    update_height(node)
    balance = get_balance(node)

    # LL
    if balance > 1 and get_balance(node.left) >= 0:
        if rotation_log is not None:
            rotation_log.append(('LL', node.val))
        return right_rotate(node)
    # LR
    if balance > 1 and get_balance(node.left) < 0:
        if rotation_log is not None:
            rotation_log.append(('LR', node.val))
        node.left = left_rotate(node.left)
        return right_rotate(node)
    # RR
    if balance < -1 and get_balance(node.right) <= 0:
        if rotation_log is not None:
            rotation_log.append(('RR', node.val))
        return left_rotate(node)
    # RL
    if balance < -1 and get_balance(node.right) > 0:
        if rotation_log is not None:
            rotation_log.append(('RL', node.val))
        node.right = right_rotate(node.right)
        return left_rotate(node)

    return node

# ==================== 可靠的树形打印（带斜线） ====================
def get_node_label(node):
    return f"{node.val}(bf={get_balance(node)})"

def build_tree_string(node):
    """
    递归构建树的字符画表示。
    返回 (lines, width, height, root_start)
    lines: 字符串列表，每一行是树的字符画
    width: 整个画的宽度（字符数）
    height: 行数
    root_start: 根节点标签的起始列索引（在 lines[0] 中）
    """
    if not node:
        return [], 0, 0, 0

    label = get_node_label(node)
    label_len = len(label)

    # 递归构建左子树和右子树
    left_lines, left_width, left_height, left_root_start = build_tree_string(node.left)
    right_lines, right_width, right_height, right_root_start = build_tree_string(node.right)

    # 计算当前子树的宽度
    # 左右子树之间的最小间隔为 2（用于放置斜线）
    gap = 2
    width = left_width + gap + right_width

    # 计算当前根节点标签的起始列位置
    # 让根节点居中于左右子树之间
    root_start = left_width + gap // 2

    # 构建当前树的字符串行
    # 第一行：根节点标签
    first_line = ' ' * root_start + label + ' ' * (width - root_start - label_len)

    # 第二行：斜线（如果有子节点）
    second_line = []
    if left_width > 0:
        # 左斜线 '/' 的位置：左子树根节点起始列 + 左子树根节点标签长度的一半
        # 为简化，直接将斜线放在 left_root_start + left_width//2 处，但需确保对齐
        # 更简单的方法：左斜线放在父节点标签左边缘左侧一列
        slash_pos = root_start - 1
        second_line.append(' ' * slash_pos + '/')
    if right_width > 0:
        backslash_pos = root_start + label_len
        if left_width > 0:
            second_line.append(' ' * (backslash_pos - len(second_line[0]) - 1) + '\\')
        else:
            second_line.append(' ' * backslash_pos + '\\')
    second_line_str = ''.join(second_line) if second_line else ''
    # 确保第二行长度与宽度一致
    if second_line_str:
        second_line_str = second_line_str.ljust(width)

    # 后续行：合并左子树和右子树的字符画
    merged_lines = []
    max_height = max(left_height, right_height)
    for i in range(max_height):
        left_line = left_lines[i] if i < left_height else ' ' * left_width
        right_line = right_lines[i] if i < right_height else ' ' * right_width
        merged_line = left_line + ' ' * gap + right_line
        merged_lines.append(merged_line)

    # 最终所有行
    all_lines = [first_line]
    if second_line_str:
        all_lines.append(second_line_str)
    all_lines.extend(merged_lines)

    # 确保所有行宽度一致
    final_width = max(len(line) for line in all_lines)
    all_lines = [line.ljust(final_width) for line in all_lines]

    return all_lines, final_width, len(all_lines), root_start

def print_tree_visual(root):
    """打印 AVL 树的带斜线图形"""
    if not root:
        print("空树")
        return
    lines, _, _, _ = build_tree_string(root)
    for line in lines:
        print(line)

# ==================== 主程序 ====================
if __name__ == "__main__":
    sequence = [30, 20, 10, 25, 40, 35, 50]
    root = None

    for i, val in enumerate(sequence, 1):
        print(f"\n========== 插入 {val} (第{i}步) ==========")
        rotation_log = []
        root = avl_insert(root, val, rotation_log)

        if rotation_log:
            for rtype, axis in rotation_log:
                print(f"⚠️ 发生失衡，类型：{rtype}，旋转轴节点值：{axis}")
        else:
            print("✅ 未发生失衡")

        print("当前 AVL 树形态：")
        print_tree_visual(root)

    # 中序遍历验证
    def inorder(node, res):
        if node:
            inorder(node.left, res)
            res.append(node.val)
            inorder(node.right, res)

    inorder_res = []
    inorder(root, inorder_res)
    print("\n" + "=" * 50)
    print("最终树中序遍历序列：", inorder_res)
    print("BST 性质验证（升序）：", inorder_res == sorted(inorder_res))


========== 插入 30 (第1步) ==========
✅ 未发生失衡
当前 AVL 树形态：
 30(bf=0)

========== 插入 20 (第2步) ==========
✅ 未发生失衡
当前 AVL 树形态：
          30(bf=1)
         /        
 20(bf=0)         

========== 插入 10 (第3步) ==========
⚠️ 发生失衡，类型：LL，旋转轴节点值：30
当前 AVL 树形态：
          20(bf=0)  
         /       \  
 10(bf=0)   30(bf=0)

========== 插入 25 (第4步) ==========
✅ 未发生失衡
当前 AVL 树形态：
          20(bf=-1)          
         /        \          
 10(bf=0)            30(bf=1)
                    /        
            25(bf=0)         

========== 插入 40 (第5步) ==========
✅ 未发生失衡
当前 AVL 树形态：
          20(bf=-1)            
         /        \            
 10(bf=0)            30(bf=0)  
                    /       \  
            25(bf=0)   40(bf=0)

========== 插入 35 (第6步) ==========
⚠️ 发生失衡，类型：RR，旋转轴节点值：20
当前 AVL 树形态：
                     30(bf=0)           
                    /       \           
          20(bf=0)              40(bf=1)
         /       \             /        
 10(bf=0)   25(bf=0)   35(bf=0)   